In [1]:
!pip install torch transformers onnx onnxruntime pandas numpy onnxscript


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 17.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is i

In [2]:
import random
import numpy as np
import pandas as pd
import warnings
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

warnings.filterwarnings("ignore")

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 1. CONSTANTS & CONFIGURATION
# MODEL_NAME = "distilbert-base-uncased" # Fast, light backbone for edge deployment
# INTENTS = ["add_text", "search_delete", "prev_sentence_delete", "bold", "italic", "underline", "unknown"]
# INTENT_MAP = {intent: idx for idx, intent in enumerate(INTENTS)}
# MAX_LEN = 64


MODEL_NAME = "distilbert-base-uncased"  # Fast, light backbone for edge deployment



MAX_LEN = 64



In [19]:
INTENTS = [

    # document editing
    "add_text",
    "search_delete",
    "prev_sentence_delete",
    "bold",
    "italic",
    "underline",

    # navigation
    "navigate_email",
    "navigate_dm",

    # email page
    "email_compose",
    "email_send",
    "email_reply",
    "email_forward",
    "email_delete",
    "email_search",

    # dm page
    "dm_send",
    "dm_reply",
    "dm_delete",
    "dm_search",

    # AI / refinement
    "ai_refine",

    # fallback
    "unknown",
]

INTENT_MAP = {intent: idx for idx, intent in enumerate(INTENTS)}

NUM_INTENTS = len(INTENT_MAP)

print("Number of intents:", NUM_INTENTS)
print("Intent map:", INTENT_MAP)

Number of intents: 20
Intent map: {'add_text': 0, 'search_delete': 1, 'prev_sentence_delete': 2, 'bold': 3, 'italic': 4, 'underline': 5, 'navigate_email': 6, 'navigate_dm': 7, 'email_compose': 8, 'email_send': 9, 'email_reply': 10, 'email_forward': 11, 'email_delete': 12, 'email_search': 13, 'dm_send': 14, 'dm_reply': 15, 'dm_delete': 16, 'dm_search': 17, 'ai_refine': 18, 'unknown': 19}


In [5]:
"""
Synthetic command data for a page-scoped assistant + document editing.

Columns
-------
text     : raw user utterance
command  : one of the intents below
data     : the single extracted slot ("" when there is nothing to extract)

Commands
--------
Document : add_text, search_delete, prev_sentence_delete,
           bold, italic, underline
Navigation : navigate_email, navigate_dm
Email page : email_compose, email_send, email_reply, email_forward,
             email_delete, email_search
DM page    : dm_send, dm_reply, dm_delete, dm_search
Fallback   : unknown
"""

import random
import pandas as pd


def generate_synthetic_data(n_per_command=300, unknown_count=300, seed=42):
    rng = random.Random(seed)

    # ---------------------------------------------------------------- vocab
    text_samples = [
        "Hello everyone", "Good morning everyone", "This is a sample document",
        "Welcome to the meeting", "Machine learning is useful.",
        "Artificial intelligence is changing the world.", "Today is a beautiful day.",
        "Please review this document.", "The project starts tomorrow.",
        "Quarterly earnings are up 10%.", "Customer onboarding is complete.",
        "The quick brown fox jumps over the lazy dog.",
    ]

    entities = [
        "Amazon", "Google", "Microsoft", "Apple", "machine learning",
        "deep learning", "artificial intelligence", "customer onboarding",
        "financial report", "project plan", "meeting notes", "Introduction",
        "Conclusion", "important information", "customer relationship management",
        "annual report", "customer experience", "Q3 results",
    ]

    # Free-form phrases used as bold/italic/underline targets. The model must
    # see these tokens tagged B-DATA/I-DATA, otherwise it associates them only
    # with whatever other command they previously appeared in (e.g. "how are
    # you" used to appear only in the `unknown` bucket).
    freeform_phrases = [
        # conversational
        "how are you", "how are you doing", "how's it going", "how have you been",
        "good morning", "good afternoon", "good evening", "hello everyone",
        "thank you", "thanks so much", "you're welcome", "see you soon",
        "nice to meet you", "long time no see", "what's up", "how was your day",
        # short sentences
        "the server is down", "the build is green", "the deploy is ready",
        "the meeting is at noon", "please review this", "let me know",
        "this is urgent", "well done", "great job", "keep it up",
        "the deadline is friday", "we shipped the release",
        "the client is happy", "the ticket is closed", "the bug is fixed",
        # noun phrases
        "the final report", "the launch date", "next steps",
        "the customer list", "the pricing table", "our new policy",
        "the roadmap", "the quarterly review", "the design doc",
        "the API key", "the security patch", "the onboarding flow",
        # greetings / farewells
        "hello world", "good night", "see you tomorrow", "take care",
        "have a nice day", "welcome aboard",
        # idiomatic / prepositional phrases -- needed so the model sees
        # multi-word function-word sequences ("in place of it") as DATA
        "in place of it", "in place of this", "in place of that",
        "instead of it", "instead of this", "as well as it",
        "in front of it", "on top of it", "at the end of it",
        "in the middle of it", "by the way", "for now",
        "in the meantime", "on the other hand", "in other words",
        # pronoun-only targets
        "it", "this", "that", "them", "these", "those",
    ]

    # Format targets = entities + full sentences + free-form phrases.
    format_targets = entities + text_samples + freeform_phrases

    email_subjects = [
        "Q3 results", "the project plan", "meeting notes", "the annual report",
        "customer onboarding", "budget approval", "the sprint review",
        "contract renewal", "the security update", "roadmap planning",
    ]

    message_bodies = [
        "the build is green", "standup moved to 10am", "I'll be late today",
        "can you review my PR", "the client call is confirmed",
        "we hit the quarterly target", "the deploy is scheduled for tonight",
        "please update the ticket", "lunch at noon", "the report is ready",
        "let's sync after the meeting", "the server is back up",
    ]

    rows = []

    def add(text, command, data=""):
        rows.append({"text": text, "command": command, "data": data})

    # ======================================================= DOCUMENT EDITING ==
    # Add text
    add_templates = [
        "{}", "Please write {}", "Write {}", "Type {}", "Add {}",
        "Insert {}", "Put {} into the document",
    ]
    for _ in range(n_per_command):
        val = rng.choice(text_samples)
        add(rng.choice(add_templates).format(val), "add_text", data=val)

    # Search & delete a phrase
    delete_templates = [
        "Delete {}", "Please remove {}", "Erase {}",
        "Delete the phrase {}", 'Delete "{}"',
    ]
    for _ in range(n_per_command):
        val = rng.choice(entities)
        add(rng.choice(delete_templates).format(val), "search_delete", data=val)

    # Delete the previous sentence
    prev_templates = [
        "Delete the previous sentence",
        "Remove the previous sentence",
        "Delete the last sentence",
    ]
    for _ in range(n_per_command):
        add(rng.choice(prev_templates), "prev_sentence_delete", data="")

    # Formatting: bold / italic / underline.
    # Includes lowercase variants ("make X bold") and the "make X to bold"
    # / "make X into bold" patterns, so the "to" / "into" tokens that sit
    # *between* the target and the format word are supervised as part of
    # the DATA span rather than as boundary markers.
    format_templates = {
        "bold": [
            "Make {} bold", "Bold {}", 'Make "{}" bold',
            "Please make {} bold", "Set {} to bold",
            "make {} bold",
            "make {} to bold",
            "make {} into bold",
            "set {} to bolded",
        ],
        "italic": [
            "Make {} italic", "Italicize {}", 'Make "{}" italic',
            "Please make {} italic", "Set {} to italic",
            "make {} italic",
            "make {} to italic",
            "make {} into italic",
            "set {} to italicized",
        ],
        "underline": [
            "Underline {}", "Apply underline to {}", 'Underline "{}"',
            "Please underline {}", "Set {} to underlined",
            "underline {}",
            "make {} to underlined",
            "make {} into underlined",
            "set {} to underline",
        ],
    }
    # Formatting needs extra coverage since the target pool is large.
    format_n = n_per_command * 2
    for command, templates in format_templates.items():
        for _ in range(format_n):
            val = rng.choice(format_targets)
            add(rng.choice(templates).format(val), command, data=val)

    # =========================================================== NAVIGATION ==
    navigate_email_templates = [
        "Open my email", "Go to email", "Take me to my inbox",
        "Open the email page", "Navigate to email", "Show me my emails",
        "Switch to email", "Go to my inbox", "Open my inbox",
        "Can you open email for me?",
    ]
    for _ in range(n_per_command):
        add(rng.choice(navigate_email_templates), "navigate_email", data="")

    navigate_dm_templates = [
        "Open my messages", "Go to chat", "Take me to my DMs",
        "Open the chat page", "Navigate to messages", "Show me my chats",
        "Switch to messages", "Go to my inbox of messages", "Open the messenger",
        "Can you open chat for me?",
    ]
    for _ in range(n_per_command):
        add(rng.choice(navigate_dm_templates), "navigate_dm", data="")

    # ========================================================== EMAIL PAGE ==
    compose_templates = [
        "Compose an email about {s}", "Write an email about {s}",
        "Start a new email with the subject {s}", "Draft an email about {s}",
        "New email about {s}", "Compose an email with the subject {s}",
        "Begin a new email regarding {s}", "Write a new email titled {s}",
    ]
    for _ in range(n_per_command):
        subj = rng.choice(email_subjects)
        add(rng.choice(compose_templates).format(s=subj), "email_compose", data=subj)

    send_templates = [
        "Send this email saying {b}", "Send the email with the message {b}",
        "Send it and say {b}", "Send the email: {b}",
        "Finish and send with the text {b}", "Send this email, the body is {b}",
        "Send the email saying {b}",
    ]
    for _ in range(n_per_command):
        body = rng.choice(message_bodies)
        add(rng.choice(send_templates).format(b=body), "email_send", data=body)

    reply_templates = [
        "Reply to this email saying {b}", "Reply with {b}",
        "Respond to this email with {b}", "Reply to the email: {b}",
        "Send a reply saying {b}", "Answer this email with {b}",
        "Reply to this email: {b}",
    ]
    for _ in range(n_per_command):
        body = rng.choice(message_bodies)
        add(rng.choice(reply_templates).format(b=body), "email_reply", data=body)

    forward_templates = [
        "Forward this email", "Forward this email to someone",
        "Forward the current email", "Send this email on",
        "Forward it", "Forward this message",
    ]
    for _ in range(n_per_command):
        add(rng.choice(forward_templates), "email_forward", data="")

    delete_email_templates = [
        "Delete this email", "Remove this email", "Trash this email",
        "Delete the current email", "Throw this email away",
        "Delete this message", "Move this email to trash",
    ]
    for _ in range(n_per_command):
        add(rng.choice(delete_email_templates), "email_delete", data="")

    search_email_templates = [
        "Search my email for {e}", "Find emails about {e}",
        "Search for {e} in my inbox", "Look up emails about {e}",
        "Find the email about {e}", "Search my inbox for {e}",
        "Show me emails about {e}",
    ]
    for _ in range(n_per_command):
        val = rng.choice(entities)
        add(rng.choice(search_email_templates).format(e=val), "email_search", data=val)

    # ============================================================= DM PAGE ==
    dm_send_templates = [
        "Send a message saying {b}", "Send a chat message: {b}",
        "Say {b}", "Write {b}", "Send {b}", "Post the message {b}",
        "Send this message: {b}", "Type and send {b}", "Message saying {b}",
    ]
    for _ in range(n_per_command):
        body = rng.choice(message_bodies)
        add(rng.choice(dm_send_templates).format(b=body), "dm_send", data=body)

    dm_reply_templates = [
        "Reply saying {b}", "Reply with {b}", "Respond with {b}",
        "Reply to this chat with {b}", "Send a reply saying {b}",
        "Answer with {b}", "Reply to this message: {b}",
    ]
    for _ in range(n_per_command):
        body = rng.choice(message_bodies)
        add(rng.choice(dm_reply_templates).format(b=body), "dm_reply", data=body)

    dm_delete_templates = [
        "Delete this chat", "Remove this message",
        "Delete the current conversation", "Trash this chat",
        "Delete this conversation", "Remove the chat", "Delete this thread",
    ]
    for _ in range(n_per_command):
        add(rng.choice(dm_delete_templates), "dm_delete", data="")

    dm_search_templates = [
        "Search my chats for {e}", "Find the chat about {e}",
        "Search messages for {e}", "Look up chats about {e}",
        "Find conversations about {e}", "Search my DMs for {e}",
        "Show me chats about {e}",
    ]
    for _ in range(n_per_command):
        val = rng.choice(entities)
        add(rng.choice(dm_search_templates).format(e=val), "dm_search", data=val)

    # ============================================================= UNKNOWN ==
    # NOTE: "How are you doing?" removed — those tokens now live only in
    # freeform_phrases so they are unambiguously bold/italic/underline targets.
    unknown_templates = [
        "What is the weather today?", "Tell me a joke",
        "What time is it?", "Who won the game?",
        "Book a table for two", "Play some music",
        "Delete it", "Send it", "Call {u}",
        "Schedule a meeting", "What's on my calendar?",
        "Mark everything as read", "Archive this thread", "Unsend that",
    ]
    fake_users = ["alice", "bob", "carol", "dave", "erin"]
    for _ in range(unknown_count):
        add(rng.choice(unknown_templates).format(u=rng.choice(fake_users)),
            "unknown", data="")

    df = pd.DataFrame(rows)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


df = generate_synthetic_data(n_per_command=400, unknown_count=400, seed=SEED)
print("Total rows:", len(df))
print(df["command"].value_counts().to_string())
print(df.sample(10, random_state=0).to_string(index=False))

Total rows: 8800
command
italic                  800
bold                    800
underline               800
email_send              400
search_delete           400
add_text                400
dm_search               400
navigate_dm             400
email_search            400
prev_sentence_delete    400
dm_send                 400
email_compose           400
email_reply             400
email_forward           400
navigate_email          400
unknown                 400
dm_reply                400
email_delete            400
dm_delete               400
                                              text       command                         data
   Send this message: let's sync after the meeting       dm_send let's sync after the meeting
Send the email with the message the build is green    email_send           the build is green
                          Italicize in other words        italic               in other words
                          Bold in the middle of it          bold   

In [20]:
# Load pre-generated synthetic dataset
CSV_PATH = "/content/synthetic_command_data_refine.csv"


df = pd.read_csv(CSV_PATH)

print("Total rows:", len(df))

print(df["command"].value_counts().to_string())

print(df.sample(10, random_state=0).to_string(index=False))

Total rows: 10000
command
ai_refine               1200
underline                800
italic                   800
bold                     800
search_delete            400
email_send               400
email_delete             400
navigate_dm              400
email_compose            400
unknown                  400
email_forward            400
dm_send                  400
prev_sentence_delete     400
dm_search                400
navigate_email           400
email_reply              400
dm_delete                400
add_text                 400
email_search             400
dm_reply                 400
                                             text      command                                data          category
   Reply with the deploy is scheduled for tonight     dm_reply the deploy is scheduled for tonight               NaN
Summarise the key points of points for discussion    ai_refine            summarise the key points discussion points
Put Machine learning is useful. into the do

In [23]:
# ------------------------------------------------------------------------------
# STEP 2: TOKENIZATION & NER + CATEGORY ALIGNMENT
# ------------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class CommandDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        text = str(row["text"])
        command = str(row["command"])

        # DATA is optional
        data_str = (
            str(row["data"])
            if pd.notna(row["data"]) and str(row["data"]).strip()
            else ""
        )

        # CATEGORY is optional
        category_str = (
            str(row["category"]).strip()
            if "category" in self.df.columns
            and pd.notna(row["category"])
            and str(row["category"]).strip()
            else ""
        )

        # ------------------------------------------------------------------
        # TOKENIZATION
        # ------------------------------------------------------------------
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt"
        )

        # ------------------------------------------------------------------
        # 1. INTENT / COMMAND LABEL
        # ------------------------------------------------------------------
        intent_label = INTENT_MAP[command]

        # ------------------------------------------------------------------
        # 2. NER / DATA LABEL
        #
        # 0 = O
        # 1 = B-DATA
        # 2 = I-DATA
        # ------------------------------------------------------------------
        ner_labels = np.zeros(self.max_len, dtype=int)

        if data_str and data_str in text:

            start_char = text.find(data_str)
            end_char = start_char + len(data_str)

            offsets = encoding["offset_mapping"][0].numpy()

            first_match = True

            for i, (start, end) in enumerate(offsets):

                # Ignore special/padding tokens
                token_id = encoding["input_ids"][0][i].item()

                if token_id in [
                    self.tokenizer.cls_token_id,
                    self.tokenizer.sep_token_id,
                    self.tokenizer.pad_token_id
                ]:
                    continue

                # Token completely inside DATA span
                if (
                    start >= start_char
                    and end <= end_char
                    and start < end
                ):
                    if first_match:
                        ner_labels[i] = 1       # B-DATA
                        first_match = False
                    else:
                        ner_labels[i] = 2       # I-DATA

        # ------------------------------------------------------------------
        # 3. CATEGORY LABEL
        #
        # category is optional.
        #
        # If category exists:
        #     category_label = CATEGORY_MAP[category]
        #
        # If category is missing:
        #     category_label = -1
        #
        # The training loss should use ignore_index=-1.
        # ------------------------------------------------------------------
        if category_str:
            category_label = CATEGORY_MAP[category_str]
        else:
            category_label = -1

        # ------------------------------------------------------------------
        # RETURN
        # ------------------------------------------------------------------
        item = {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),

            # Intent / command
            "intent_labels": torch.tensor(
                intent_label,
                dtype=torch.long
            ),

            # DATA NER
            "ner_labels": torch.tensor(
                ner_labels,
                dtype=torch.long
            ),

            # Category classification
            "category_labels": torch.tensor(
                category_label,
                dtype=torch.long
            )
        }

        return item


# ------------------------------------------------------------------------------
# TRAIN / VALIDATION SPLIT
# ------------------------------------------------------------------------------

split = int(len(df) * 0.8)

train_df = df.iloc[:split].reset_index(drop=True)
val_df = df.iloc[split:].reset_index(drop=True)


# ------------------------------------------------------------------------------
# DATASETS
# ------------------------------------------------------------------------------

train_dataset = CommandDataset(
    train_df,
    tokenizer,
    MAX_LEN
)

val_dataset = CommandDataset(
    val_df,
    tokenizer,
    MAX_LEN
)


# ------------------------------------------------------------------------------
# DATALOADERS
# ------------------------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

In [25]:
# ------------------------------------------------------------------------------
# CATEGORY LABEL MAPPING
# ------------------------------------------------------------------------------

# Make sure category column exists
if "category" not in df.columns:
    raise ValueError(
        "The dataframe must contain a 'category' column."
    )

# Get non-empty categories
categories = sorted(
    c
    for c in df["category"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    if c
)

# Category -> integer ID
CATEGORY_MAP = {
    category: idx
    for idx, category in enumerate(categories)
}

# Number of category classes
NUM_CATEGORIES = len(CATEGORY_MAP)

print("CATEGORY_MAP:")
for category, idx in CATEGORY_MAP.items():
    print(f"  {idx}: {category}")

print(f"\nNUM_CATEGORIES = {NUM_CATEGORIES}")

CATEGORY_MAP:
  0: discussion points
  1: key meeting objectives
  2: pipeline deals
  3: recent relationship developments
  4: relationship sensitivities

NUM_CATEGORIES = 5


In [26]:
# ------------------------------------------------------------------------------
# STEP 3: JOINT MULTI-TASK ARCHITECTURE
# Intent + NER + Category Classification
# ------------------------------------------------------------------------------

from transformers import AutoModel


class JointIntentNERCategoryModel(nn.Module):

    def __init__(
        self,
        model_name,
        num_intents,
        num_categories,
        num_ner_tags=3
    ):
        super(JointIntentNERCategoryModel, self).__init__()

        # ------------------------------------------------------------------
        # Shared DistilBERT encoder
        # ------------------------------------------------------------------
        self.transformer = AutoModel.from_pretrained(model_name)

        hidden_size = self.transformer.config.hidden_size

        # ------------------------------------------------------------------
        # HEAD 1: Intent / Command Classification
        # Example:
        #   add_text
        #   bold
        #   email_send
        #   ai_refine
        # ------------------------------------------------------------------
        self.intent_classifier = nn.Linear(
            hidden_size,
            num_intents
        )

        # ------------------------------------------------------------------
        # HEAD 2: DATA NER / Token Classification
        #
        # 0 = O
        # 1 = B-DATA
        # 2 = I-DATA
        # ------------------------------------------------------------------
        self.ner_classifier = nn.Linear(
            hidden_size,
            num_ner_tags
        )

        # ------------------------------------------------------------------
        # HEAD 3: Category Classification
        #
        # Example:
        #   key meeting objectives
        #   pipeline deals
        #   relationship sensitivities
        #   discussion points
        # ------------------------------------------------------------------
        self.category_classifier = nn.Linear(
            hidden_size,
            num_categories
        )

    def forward(self, input_ids, attention_mask):

        # Shared Transformer
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = outputs.last_hidden_state
        # [batch_size, seq_len, hidden_size]

        # CLS representation
        pooled_output = sequence_output[:, 0, :]
        # [batch_size, hidden_size]

        # ------------------------------------------------------------------
        # Intent classification
        # ------------------------------------------------------------------
        intent_logits = self.intent_classifier(
            pooled_output
        )

        # ------------------------------------------------------------------
        # NER / DATA extraction
        # ------------------------------------------------------------------
        ner_logits = self.ner_classifier(
            sequence_output
        )

        # ------------------------------------------------------------------
        # Category classification
        # ------------------------------------------------------------------
        category_logits = self.category_classifier(
            pooled_output
        )

        return (
            intent_logits,
            ner_logits,
            category_logits
        )


# ------------------------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ------------------------------------------------------------------------------
# MODEL
# ------------------------------------------------------------------------------

model = JointIntentNERCategoryModel(
    model_name=MODEL_NAME,
    num_intents=len(INTENTS),
    num_categories=NUM_CATEGORIES
).to(device)


print(model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


JointIntentNERCategoryModel(
  (transformer): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
       

In [27]:
# ------------------------------------------------------------------------------
# STEP 4: MODEL TRAINING LOOP
# Intent + NER + Category
# ------------------------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5
)

# Intent loss
criterion_intent = nn.CrossEntropyLoss()

# NER loss
criterion_ner = nn.CrossEntropyLoss()

# Category loss
#
# category_labels = -1 when category is missing.
# ignore_index=-1 means those samples do not contribute
# to the category loss.
criterion_category = nn.CrossEntropyLoss(
    ignore_index=-1
)


print("\nTraining Joint Model...")

model.train()

for epoch in range(3):

    total_loss = 0
    total_intent_loss = 0
    total_ner_loss = 0
    total_category_loss = 0

    for batch in train_loader:

        optimizer.zero_grad()

        # ------------------------------------------------------------------
        # INPUTS
        # ------------------------------------------------------------------
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # ------------------------------------------------------------------
        # LABELS
        # ------------------------------------------------------------------
        intent_labels = batch["intent_labels"].to(device)
        ner_labels = batch["ner_labels"].to(device)
        category_labels = batch["category_labels"].to(device)

        # ------------------------------------------------------------------
        # MODEL FORWARD
        # ------------------------------------------------------------------
        (
            intent_logits,
            ner_logits,
            category_logits
        ) = model(
            input_ids,
            attention_mask
        )

        # ------------------------------------------------------------------
        # INTENT LOSS
        # ------------------------------------------------------------------
        loss_intent = criterion_intent(
            intent_logits,
            intent_labels
        )

        # ------------------------------------------------------------------
        # NER LOSS
        #
        # ner_logits:
        # [batch_size, seq_len, 3]
        #
        # Convert to:
        # [batch_size * seq_len, 3]
        # ------------------------------------------------------------------
        loss_ner = criterion_ner(
            ner_logits.view(-1, 3),
            ner_labels.view(-1)
        )

        # ------------------------------------------------------------------
        # CATEGORY LOSS
        #
        # category_labels:
        #   0 ... NUM_CATEGORIES-1 -> valid category
        #   -1                      -> category not provided
        #
        # CrossEntropyLoss(ignore_index=-1) ignores missing categories.
        # ------------------------------------------------------------------
        loss_category = criterion_category(
            category_logits,
            category_labels
        )

        # ------------------------------------------------------------------
        # COMBINED MULTI-TASK LOSS
        # ------------------------------------------------------------------
        loss = (
            loss_intent
            + loss_ner
            + loss_category
        )

        # ------------------------------------------------------------------
        # BACKPROPAGATION
        # ------------------------------------------------------------------
        loss.backward()

        optimizer.step()

        # ------------------------------------------------------------------
        # TRACK LOSSES
        # ------------------------------------------------------------------
        total_loss += loss.item()
        total_intent_loss += loss_intent.item()
        total_ner_loss += loss_ner.item()
        total_category_loss += loss_category.item()

    # ----------------------------------------------------------------------
    # EPOCH RESULTS
    # ----------------------------------------------------------------------
    num_batches = len(train_loader)

    print(
        f"Epoch {epoch + 1} Complete | "
        f"Total Loss: {total_loss / num_batches:.4f} | "
        f"Intent: {total_intent_loss / num_batches:.4f} | "
        f"NER: {total_ner_loss / num_batches:.4f} | "
        f"Category: {total_category_loss / num_batches:.4f}"
    )


Training Joint Model...
Epoch 1 Complete | Total Loss: nan | Intent: 0.5365 | NER: 0.0753 | Category: nan
Epoch 2 Complete | Total Loss: nan | Intent: 0.0337 | NER: 0.0094 | Category: nan
Epoch 3 Complete | Total Loss: nan | Intent: 0.0246 | NER: 0.0022 | Category: nan


In [28]:
# ------------------------------------------------------------------------------
# STEP 4.5: TEST AND EVALUATE THE MODEL
# Intent + NER + Category
# ------------------------------------------------------------------------------

print("\nEvaluating model performance on validation dataset...")

model.eval()

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

# ------------------------------------------------------------------------------
# INTENT METRICS
# ------------------------------------------------------------------------------

correct_intents = 0
total_intent_samples = 0

# ------------------------------------------------------------------------------
# NER METRICS
# ------------------------------------------------------------------------------

correct_ner_tokens = 0
total_ner_tokens = 0

# ------------------------------------------------------------------------------
# CATEGORY METRICS
# ------------------------------------------------------------------------------

correct_categories = 0
total_category_samples = 0


# ------------------------------------------------------------------------------
# INFERENCE
# ------------------------------------------------------------------------------

with torch.no_grad():

    for batch in val_loader:

        # ----------------------------------------------------------------------
        # INPUTS
        # ----------------------------------------------------------------------

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # ----------------------------------------------------------------------
        # LABELS
        # ----------------------------------------------------------------------

        intent_labels = batch["intent_labels"].to(device)
        ner_labels = batch["ner_labels"].to(device)
        category_labels = batch["category_labels"].to(device)

        # ----------------------------------------------------------------------
        # MODEL
        # ----------------------------------------------------------------------

        (
            intent_logits,
            ner_logits,
            category_logits
        ) = model(
            input_ids,
            attention_mask
        )

        # ======================================================================
        # 1. INTENT / COMMAND ACCURACY
        # ======================================================================

        intent_preds = torch.argmax(
            intent_logits,
            dim=-1
        )

        correct_intents += (
            intent_preds == intent_labels
        ).sum().item()

        total_intent_samples += intent_labels.size(0)

        # ======================================================================
        # 2. NER / DATA TOKEN ACCURACY
        # ======================================================================

        ner_preds = torch.argmax(
            ner_logits,
            dim=-1
        )

        # ----------------------------------------------------------------------
        # Only evaluate actual tokens.
        #
        # attention_mask = 1
        # excludes padding.
        #
        # Also exclude CLS and SEP tokens because they should not be
        # considered DATA/O tokens during evaluation.
        # ----------------------------------------------------------------------

        active_mask = attention_mask.bool()

        # Exclude CLS tokens
        if tokenizer.cls_token_id is not None:
            active_mask &= (
                input_ids != tokenizer.cls_token_id
            )

        # Exclude SEP tokens
        if tokenizer.sep_token_id is not None:
            active_mask &= (
                input_ids != tokenizer.sep_token_id
            )

        active_labels = ner_labels[active_mask]
        active_preds = ner_preds[active_mask]

        correct_ner_tokens += (
            active_preds == active_labels
        ).sum().item()

        total_ner_tokens += active_labels.numel()

        # ======================================================================
        # 3. CATEGORY ACCURACY
        # ======================================================================

        category_preds = torch.argmax(
            category_logits,
            dim=-1
        )

        # ----------------------------------------------------------------------
        # category_labels == -1 means category was not supplied.
        #
        # These samples must NOT be included in category accuracy.
        # ----------------------------------------------------------------------

        category_mask = category_labels != -1

        active_category_labels = category_labels[
            category_mask
        ]

        active_category_preds = category_preds[
            category_mask
        ]

        correct_categories += (
            active_category_preds == active_category_labels
        ).sum().item()

        total_category_samples += (
            category_mask.sum().item()
        )


# ------------------------------------------------------------------------------
# CALCULATE METRICS
# ------------------------------------------------------------------------------

intent_accuracy = (
    correct_intents / total_intent_samples * 100
    if total_intent_samples > 0
    else 0.0
)

ner_accuracy = (
    correct_ner_tokens / total_ner_tokens * 100
    if total_ner_tokens > 0
    else 0.0
)

category_accuracy = (
    correct_categories / total_category_samples * 100
    if total_category_samples > 0
    else 0.0
)


# ------------------------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

print(
    f"Intent Classification Accuracy: {intent_accuracy:.2f}%"
)

print(
    f"NER Token-Level Accuracy:       {ner_accuracy:.2f}%"
)

print(
    f"Category Classification Accuracy: {category_accuracy:.2f}%"
)

print("-" * 70)

print(
    f"Intent samples evaluated:       {total_intent_samples}"
)

print(
    f"NER tokens evaluated:            {total_ner_tokens}"
)

print(
    f"Category samples evaluated:      {total_category_samples}"
)

print(
    f"Category samples ignored:        "
    f"{len(val_dataset) - total_category_samples}"
)

print("=" * 70)


Evaluating model performance on validation dataset...

VALIDATION RESULTS
Intent Classification Accuracy: 99.05%
NER Token-Level Accuracy:       99.98%
Category Classification Accuracy: 100.00%
----------------------------------------------------------------------
Intent samples evaluated:       2000
NER tokens evaluated:            12634
Category samples evaluated:      241
Category samples ignored:        1759


In [30]:
# ------------------------------------------------------------------------------
# QUICK MANUAL TEST VERIFICATION
# Intent + NER/Data + Category
# ------------------------------------------------------------------------------

print("\nRunning quick manual verification test...")

test_phrase = "make in place of it to bold"


# ------------------------------------------------------------------------------
# TOKENIZE TEST INPUT
# ------------------------------------------------------------------------------

inputs = tokenizer(
    test_phrase,
    return_tensors="pt",
    max_length=MAX_LEN,
    padding="max_length",
    truncation=True
)


# ------------------------------------------------------------------------------
# MODEL INFERENCE
# ------------------------------------------------------------------------------

model.eval()

with torch.no_grad():

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # Model now returns THREE outputs
    (
        i_log,
        n_log,
        c_log
    ) = model(
        input_ids,
        attention_mask
    )


    # ==========================================================================
    # 1. DECODE INTENT / COMMAND
    # ==========================================================================

    intent_probs = torch.softmax(
        i_log,
        dim=-1
    ).squeeze(0)

    predicted_intent_idx = torch.argmax(
        intent_probs
    ).item()

    intent_confidence = intent_probs[
        predicted_intent_idx
    ].item()

    predicted_intent = INTENTS[
        predicted_intent_idx
    ]


    # ==========================================================================
    # 2. DECODE NER / DATA
    # ==========================================================================

    ner_tags = torch.argmax(
        n_log,
        dim=-1
    ).squeeze(0).cpu().numpy()

    input_token_ids = inputs[
        "input_ids"
    ].squeeze(0).cpu().tolist()

    tokens = tokenizer.convert_ids_to_tokens(
        input_token_ids
    )

    extracted_tokens = []

    for token, tag in zip(tokens, ner_tags):

        # Skip special and padding tokens
        if token in [
            tokenizer.cls_token,
            tokenizer.sep_token,
            tokenizer.pad_token
        ]:
            continue

        # B-DATA = 1
        # I-DATA = 2
        if tag > 0:

            if token.startswith("##"):

                # WordPiece continuation
                cleaned_token = token[2:]

                if extracted_tokens:
                    extracted_tokens[-1] += cleaned_token
                else:
                    extracted_tokens.append(cleaned_token)

            else:
                extracted_tokens.append(token)

    extracted_data = " ".join(
        extracted_tokens
    ).strip()


    # ==========================================================================
    # 3. DECODE CATEGORY
    # ==========================================================================

    category_probs = torch.softmax(
        c_log,
        dim=-1
    ).squeeze(0)

    predicted_category_idx = torch.argmax(
        category_probs
    ).item()

    category_confidence = category_probs[
        predicted_category_idx
    ].item()

    # CATEGORY_MAP is:
    # {
    #     "discussion points": 0,
    #     "key meeting objectives": 1,
    #     ...
    # }
    #
    # We need the reverse mapping.
    IDX_TO_CATEGORY = {
        idx: category
        for category, idx in CATEGORY_MAP.items()
    }

    predicted_category = IDX_TO_CATEGORY.get(
        predicted_category_idx,
        "unknown"
    )


# ------------------------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------------------------

print("\n" + "=" * 70)
print("MANUAL TEST RESULT")
print("=" * 70)

print(f"Input text:          '{test_phrase}'")

print(
    f"Predicted Command:   {predicted_intent} "
    f"(Confidence: {intent_confidence:.2%})"
)

print(
    f"Extracted Data:      '{extracted_data}'"
)

print(
    f"Predicted Category:  {predicted_category} "
    f"(Confidence: {category_confidence:.2%})"
)

print("=" * 70)


Running quick manual verification test...

MANUAL TEST RESULT
Input text:          'make in place of it to bold'
Predicted Command:   bold (Confidence: 99.89%)
Extracted Data:      'in place of it'
Predicted Category:  recent relationship developments (Confidence: 40.07%)


In [31]:
# ------------------------------------------------------------------------------
# EXPORT TO ONNX FORMAT
# ------------------------------------------------------------------------------
print("\nExporting model architecture to ONNX format...")

model.eval()

# ------------------------------------------------------------------------------
# Representative dummy inputs
# ------------------------------------------------------------------------------
dummy_input_ids = torch.ones(
    1,
    MAX_LEN,
    dtype=torch.long
).to(device)

dummy_attention_mask = torch.ones(
    1,
    MAX_LEN,
    dtype=torch.long
).to(device)

onnx_model_path = "joint_command_parser.onnx"


# ------------------------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------------------------
torch.onnx.export(
    model,
    (
        dummy_input_ids,
        dummy_attention_mask
    ),
    f=onnx_model_path,

    # Inputs
    input_names=[
        "input_ids",
        "attention_mask"
    ],

    # Model now has THREE outputs
    output_names=[
        "intent_logits",
        "ner_logits",
        "category_logits"
    ],

    # Dynamic batch dimension
    dynamic_axes={
        "input_ids": {
            0: "batch_size"
        },

        "attention_mask": {
            0: "batch_size"
        },

        "intent_logits": {
            0: "batch_size"
        },

        "ner_logits": {
            0: "batch_size"
        },

        "category_logits": {
            0: "batch_size"
        }
    },

    opset_version=11
)

print(f"\nONNX model exported successfully: {onnx_model_path}")

W1005 14:17:13.687000 1145 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features



Exporting model architecture to ONNX format...
[torch.onnx] Obtain model graph for `JointIntentNERCategoryModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `JointIntentNERCategoryModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/no_previous_version.h:23: adap

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

ONNX model exported successfully: joint_command_parser.onnx


In [32]:
import onnx, os

# Load fully — this pulls in the .data file
m = onnx.load("joint_command_parser.onnx")

# Sanity check that weights really came in
print("initializers loaded:", len(m.graph.initializer))
print("first initializer dims:",
      m.graph.initializer[0].name,
      list(m.graph.initializer[0].dims))

# Save as one self-contained file
onnx.save_model(
    m,
    "joint_command_parser_single.onnx",
    save_as_external_data=False
)

size = os.path.getsize("joint_command_parser_single.onnx")
print("single-file size (MB):", round(size / 1024 / 1024, 1))

initializers loaded: 125
first initializer dims: transformer.embeddings.LayerNorm.weight [768]
single-file size (MB): 253.9


In [33]:
import onnx

m = onnx.load("joint_command_parser_single.onnx")

# Drop stale value_info entries (and any intermediate shape hints)
m.graph.ClearField("value_info")

# Also clear dim_param annotations on inputs/outputs that might conflict
for vi in list(m.graph.input) + list(m.graph.output):
    if vi.type.HasField("tensor_type"):
        t = vi.type.tensor_type
        for d in t.shape.dim:
            if d.HasField("dim_value") and d.dim_value <= 0:
                d.ClearField("dim_value")
                d.dim_param = "dynamic"

onnx.save_model(
    m,
    "joint_command_parser_clean.onnx",
    save_as_external_data=False
)

print("saved cleaned model")

saved cleaned model


In [34]:
import onnx
m2 = onnx.load("joint_command_parser_clean.onnx", load_external_data=False)
print("nodes:", len(m2.graph.node))
print("value_info entries (should be 0):", len(m2.graph.value_info))

nodes: 301
value_info entries (should be 0): 0


In [35]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="joint_command_parser_clean.onnx",
    model_output="joint_command_parser_int8.onnx",
    weight_type=QuantType.QInt8,
    extra_options={
        # Skip the internal shape inference pass that's tripping on the
        # stale annotation
        'DisableShapeInference': True,
    }
)

In [36]:
import os, onnx
p = "joint_command_parser_int8.onnx"
print("int8 size (MB):", round(os.path.getsize(p) / 1024 / 1024, 1))

m = onnx.load(p, load_external_data=False)
ext = [t.external_data for t in m.graph.initializer if t.external_data]
print("external_data entries (must be 0):", len(ext))

int8 size (MB): 64.1
external_data entries (must be 0): 0


In [ ]:
!pip freeze > requirements.txt


In [ ]:
!pip freeze

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

# 1. Fetch token securely from Colab Secrets
try:
    TOKEN = userdata.get('HF_TOKEN')
except Exception:
    raise ValueError("Could not find 'HF_TOKEN' in your Colab Secrets.")

api = HfApi(token=TOKEN)

# 2. AUTOMATIC USERNAME LOOKUP
# This grabs your actual Hugging Face profile information using the token
try:
    user_info = api.whoami()
    USERNAME = user_info['name']
    print(f"Authenticated successfully as user: '{USERNAME}'")
except Exception as e:
    raise ValueError(f"Failed to authenticate with token. Check permissions: {e}")

# 3. Configuration (Uses your automated true username)
REPO_NAME = "distilbert-command-data-tagger"
REPO_ID = f"{USERNAME}/{REPO_NAME}"

# 4. Create the remote repository
print(f"Checking/Creating repository: {REPO_ID}...")
create_repo(repo_id=REPO_ID, repo_type="model", token=TOKEN, exist_ok=True)

# 5. Upload your ONNX structural file
print("Uploading joint_command_parser.onnx...")
api.upload_file(
    path_or_fileobj="joint_command_parser.onnx",
    path_in_repo="joint_command_parser.onnx",
    repo_id=REPO_ID
)

# 6. Upload the associated network weights binary
print("Uploading joint_command_parser.onnx.data...")
api.upload_file(
    path_or_fileobj="joint_command_parser.onnx.data",
    path_in_repo="joint_command_parser.onnx.data",
    repo_id=REPO_ID
)

print(f"\nAll assets pushed! View your files live at: https://huggingface.co/{REPO_ID}/tree/main")


In [ ]:
import os

handler_code = """
import os
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

class EndpointHandler:
    def __init__(self, path=""):
        # 1. Resolve paths for BOTH structural graph and matrix weights
        model_path = os.path.join(path, "joint_command_parser.onnx")

        # Initialize the ONNX Runtime execution engine thread pool
        self.session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])

        # 2. Match the exact tokenizer backbone used during training
        self.tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

        # 3. Structural target layouts
        self.intents = [
            # document editing
            "add_text",
            "search_delete",
            "prev_sentence_delete",
            "bold",
            "italic",
            "underline",
            # navigation
            "navigate_email",
            "navigate_dm",
            # email page
            "email_compose",
            "email_send",
            "email_reply",
            "email_forward",
            "email_delete",
            "email_search",
            # dm page
            "dm_send",
            "dm_reply",
            "dm_delete",
            "dm_search",
            # fallback
            "unknown",
        ]
        self.max_len = 64

    def __call__(self, data):
        # Extract input text payload sent via HTTP POST Request
        inputs_payload = data.get("inputs", "")
        if not inputs_payload:
            return {"error": "Missing 'inputs' string parameter inside payload."}

        # Tokenize incoming sequence matching matrix bounds
        tokenized = self.tokenizer(
            inputs_payload,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="np"
        )

        onnx_feeds = {
            "input_ids": tokenized["input_ids"].astype(np.int64),
            "attention_mask": tokenized["attention_mask"].astype(np.int64)
        }

        # Run execution graph matrix calculations
        intent_logits, ner_logits = self.session.run(None, onnx_feeds)

        # Decode Intent array using Softmax calculation
        logits_exp = np.exp(intent_logits[0])
        probs = logits_exp / np.sum(logits_exp)
        intent_idx = np.argmax(probs)

        # Apply strict fallback thresholds
        confidence = float(probs[intent_idx])
        final_intent = self.intents[intent_idx] if confidence >= 0.65 else "unknown"

        # Process and decode non-zero NER spans
        ner_tags = np.argmax(ner_logits[0], axis=-1)
        tokens = self.tokenizer.convert_ids_to_tokens(tokenized["input_ids"][0])

        extracted_tokens = []
        for token, tag in zip(tokens, ner_tags):
            if token in [self.tokenizer.cls_token, self.tokenizer.sep_token, self.tokenizer.pad_token]:
                continue
            if tag > 0:  # Valid B-DATA or I-DATA sequences
                cleaned = token.replace("##", "")
                if token.startswith("##") and extracted_tokens:
                    extracted_tokens[-1] += cleaned
                else:
                    extracted_tokens.append(cleaned)

        extracted_data = " ".join(extracted_tokens).strip()

        return {
            "intent": final_intent,
            "confidence": confidence,
            "extracted_data": extracted_data
        }
"""

with open("handler.py", "w") as f:
    f.write(handler_code.strip())
print("handler.py script structured successfully!")


In [ ]:
import os
import sys

# 1. Ensure the folder containing your handler and model is visible to Python
# If your files are in the 'hf_upload_folder' directory, we point the handler path there
MODEL_DIR = "./"

if not os.path.exists(os.path.join(MODEL_DIR, "handler.py")):
    print(f"⚠️ Error: handler.py not found inside '{MODEL_DIR}'. Checking current working directory instead...")
    MODEL_DIR = "."

# Add the directory to Python's search path so we can import it
sys.path.append(os.path.abspath(MODEL_DIR))

try:
    # 2. Import the EndpointHandler from your handler file
    from handler import EndpointHandler
    print("✅ Successfully imported EndpointHandler from handler.py!")

    # 3. Initialize the handler (Simulating Hugging Face loading the model on startup)
    print("Initializing handler model session...")
    handler_instance = EndpointHandler(path=MODEL_DIR)
    print("✅ ONNX Runtime Session successfully initialized inside handler!")

    # 4. Mock a production payload dictionary
    mock_payload = {
        "inputs": 'Please make "Quarterly earnings are up 10%" bold'
    }

    # 5. Run execution test
    print("\nRunning inference test against mock payload...")
    result = handler_instance(mock_payload)

    # Print out formatted verification matrix
    print("\n--- HANDLER OUTPUT RESULT ---")
    print(f"Predicted Intent:   {result.get('intent')}")
    print(f"Confidence Score:   {result.get('confidence'):.2%}")
    print(f"Extracted Data:     '{result.get('extracted_data')}'")
    print("------------------------------")

    if result.get('intent') == "bold" and "Quarterly earnings" in result.get('extracted_data', ''):
        print("🎉 Success! Your handler.py works and parses multi-task outputs perfectly.")
    else:
        print("⚠️ Warning: Handler executed, but outputs don't match the expected test definitions.")

except ModuleNotFoundError as e:
    print(f"❌ Failed to find or import handler file: {e}")
except Exception as e:
    print(f"❌ Runtime execution crash inside handler.py: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
import shutil
os.makedirs("hf_upload_folder", exist_ok=True)
shutil.move("handler.py", "hf_upload_folder/handler.py")
shutil.move("requirements.txt", "hf_upload_folder/requirements.txt")

print(f"Pushing inference hooks directly into: {REPO_ID}...")
api.upload_folder(
    folder_path="hf_upload_folder",
    repo_id=REPO_ID,
    repo_type="model"
)
print("Upload finalized! Hub infrastructure endpoints updated.")
